In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/cleansteps

/content/drive/MyDrive/cleansteps


In [ ]:
import os

print(os.path.exists("data/splits/train.csv"))
print(os.path.exists("data/splits/valid.csv"))
print(os.path.exists("data/splits/test.csv"))

True
True
True


In [ ]:
!pip install -q transformers accelerate scikit-learn pandas matplotlib

In [ ]:
import os

os.makedirs("experiments", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

In [ ]:
%%writefile experiments/06_bert_512_baseline.py

from __future__ import annotations

import json
import random
import inspect
import logging
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)


@dataclass
class Config:
    train_path: str = "data/splits/train.csv"
    valid_path: str = "data/splits/valid.csv"
    test_path: str = "data/splits/test.csv"

    output_dir: str = "outputs/06_bert_512_baseline"

    text_col: str = "case_text"
    label_col: str = "label"

    model_name: str = "bert-base-uncased"

    max_length: int = 512
    top_k: int = 5

    random_state: int = 42

    num_train_epochs: int = 8
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 2

    train_batch_size_gpu: int = 8
    eval_batch_size_gpu: int = 8
    train_batch_size_cpu: int = 4
    eval_batch_size_cpu: int = 4

    save_total_limit: int = 2

    debug_small_run: bool = False
    debug_train_size: int = 80
    debug_valid_size: int = 40
    debug_test_size: int = 40

    @property
    def output_path(self) -> Path:
        path = Path(self.output_dir)
        path.mkdir(parents=True, exist_ok=True)
        return path

    @property
    def checkpoint_path(self) -> Path:
        path = self.output_path / "checkpoints"
        path.mkdir(parents=True, exist_ok=True)
        return path


CFG = Config()


def set_reproducibility(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    set_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


class ClinicalTextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length: int):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
        )

        encoded["labels"] = self.labels[idx]

        return encoded


def load_splits(cfg: Config):
    train_df = pd.read_csv(cfg.train_path)
    valid_df = pd.read_csv(cfg.valid_path)
    test_df = pd.read_csv(cfg.test_path)

    required_cols = [cfg.text_col, cfg.label_col]

    for split_name, df in [
        ("train", train_df),
        ("valid", valid_df),
        ("test", test_df),
    ]:
        missing = [col for col in required_cols if col not in df.columns]

        if missing:
            raise ValueError(f"{split_name} split missing columns: {missing}")

        df.dropna(subset=required_cols, inplace=True)
        df[cfg.text_col] = df[cfg.text_col].astype(str)
        df[cfg.label_col] = df[cfg.label_col].astype(str)

    if cfg.debug_small_run:
        train_df = train_df.sample(
            min(cfg.debug_train_size, len(train_df)),
            random_state=cfg.random_state,
        )

        valid_df = valid_df.sample(
            min(cfg.debug_valid_size, len(valid_df)),
            random_state=cfg.random_state,
        )

        test_df = test_df.sample(
            min(cfg.debug_test_size, len(test_df)),
            random_state=cfg.random_state,
        )

        logger.warning("DEBUG_SMALL_RUN is active.")

    logger.info("Train rows: %d", len(train_df))
    logger.info("Valid rows: %d", len(valid_df))
    logger.info("Test rows: %d", len(test_df))

    return train_df, valid_df, test_df


def build_label_mappings(train_df, valid_df, test_df, cfg: Config):
    labels = sorted(train_df[cfg.label_col].unique().tolist())

    label2id = {
        label: idx
        for idx, label in enumerate(labels)
    }

    id2label = {
        idx: label
        for label, idx in label2id.items()
    }

    for split_name, df in [
        ("valid", valid_df),
        ("test", test_df),
    ]:
        split_labels = set(df[cfg.label_col].unique())
        missing = split_labels - set(labels)

        if missing:
            raise ValueError(
                f"{split_name} contains labels not present in train: {sorted(missing)}"
            )

    logger.info("Number of classes: %d", len(label2id))

    return label2id, id2label


def save_token_length_stats(df_all, tokenizer, cfg: Config):
    lengths = []

    for text in df_all[cfg.text_col].astype(str).tolist():
        tokenized = tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
        )

        lengths.append(len(tokenized["input_ids"]))

    lengths = np.array(lengths)

    stats = {
        "count": int(len(lengths)),
        "mean": float(np.mean(lengths)),
        "median": float(np.median(lengths)),
        "p90": float(np.percentile(lengths, 90)),
        "p95": float(np.percentile(lengths, 95)),
        "p99": float(np.percentile(lengths, 99)),
        "max": int(np.max(lengths)),
        "max_length_used": cfg.max_length,
        "texts_over_max_length": int(np.sum(lengths > cfg.max_length)),
        "percent_over_max_length": float(np.mean(lengths > cfg.max_length)),
    }

    with open(
        cfg.output_path / "token_length_stats.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(stats, f, indent=4, ensure_ascii=False)

    plt.figure(figsize=(10, 6))
    plt.hist(lengths, bins=50)
    plt.axvline(cfg.max_length, linestyle="--", label=f"max_length={cfg.max_length}")
    plt.title("Token Length Distribution")
    plt.xlabel("Number of tokens")
    plt.ylabel("Number of cases")
    plt.legend()
    plt.tight_layout()
    plt.savefig(cfg.output_path / "token_length_distribution.png", dpi=300)
    plt.close()

    logger.info("Token length stats saved.")


def softmax_np(logits):
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)

    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def compute_metrics_builder(cfg: Config, num_labels: int):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred

        y_pred = np.argmax(logits, axis=1)
        probs = softmax_np(logits)

        return {
            "accuracy": accuracy_score(labels, y_pred),
            "macro_f1": f1_score(
                labels,
                y_pred,
                average="macro",
                zero_division=0,
            ),
            "weighted_f1": f1_score(
                labels,
                y_pred,
                average="weighted",
                zero_division=0,
            ),
            f"top{cfg.top_k}_accuracy": top_k_accuracy_score(
                labels,
                probs,
                k=cfg.top_k,
                labels=np.arange(num_labels),
            ),
        }

    return compute_metrics


def build_training_args(cfg: Config):
    use_gpu = torch.cuda.is_available()

    train_batch_size = (
        cfg.train_batch_size_gpu
        if use_gpu
        else cfg.train_batch_size_cpu
    )

    eval_batch_size = (
        cfg.eval_batch_size_gpu
        if use_gpu
        else cfg.eval_batch_size_cpu
    )

    logger.info("CUDA available: %s", use_gpu)

    args = {
        "output_dir": str(cfg.checkpoint_path),
        "learning_rate": cfg.learning_rate,
        "per_device_train_batch_size": train_batch_size,
        "per_device_eval_batch_size": eval_batch_size,
        "num_train_epochs": cfg.num_train_epochs,
        "weight_decay": cfg.weight_decay,
        "logging_dir": str(cfg.output_path / "logs"),
        "logging_strategy": "epoch",
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss",
        "greater_is_better": False,
        "save_total_limit": cfg.save_total_limit,
        "report_to": "none",
        "seed": cfg.random_state,
        "fp16": use_gpu,
    }

    params = inspect.signature(TrainingArguments.__init__).parameters

    if "eval_strategy" in params:
        args["eval_strategy"] = "epoch"
    else:
        args["evaluation_strategy"] = "epoch"

    return TrainingArguments(**args)


def save_training_history(trainer: Trainer, cfg: Config):
    history = trainer.state.log_history
    history_df = pd.DataFrame(history)

    history_df.to_csv(
        cfg.output_path / "training_log_history.csv",
        index=False,
    )

    train_logs = [
        item
        for item in history
        if "loss" in item and "eval_loss" not in item and "epoch" in item
    ]

    eval_logs = [
        item
        for item in history
        if "eval_loss" in item and "epoch" in item
    ]

    plt.figure(figsize=(10, 6))

    if train_logs:
        plt.plot(
            [item["epoch"] for item in train_logs],
            [item["loss"] for item in train_logs],
            marker="o",
            label="training loss",
        )

    if eval_logs:
        plt.plot(
            [item["epoch"] for item in eval_logs],
            [item["eval_loss"] for item in eval_logs],
            marker="o",
            label="validation loss",
        )

    plt.title("BERT 512 Training and Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(cfg.output_path / "train_valid_loss.png", dpi=300)
    plt.close()


def save_test_outputs(trainer, test_dataset, test_df, id2label, cfg: Config):
    prediction_output = trainer.predict(test_dataset)

    logits = prediction_output.predictions
    y_true = prediction_output.label_ids

    probs = softmax_np(logits)
    y_pred = np.argmax(probs, axis=1)

    num_labels = len(id2label)

    metrics = {
        "test_accuracy": accuracy_score(y_true, y_pred),
        "test_macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "test_weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
        f"test_top{cfg.top_k}_accuracy": top_k_accuracy_score(
            y_true,
            probs,
            k=cfg.top_k,
            labels=np.arange(num_labels),
        ),
    }

    with open(
        cfg.output_path / "test_metrics.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(metrics, f, indent=4, ensure_ascii=False)

    topk_indices = np.argsort(probs, axis=1)[:, ::-1][:, :cfg.top_k]
    topk_probs = np.take_along_axis(probs, topk_indices, axis=1)

    rows = []

    for i in range(len(test_df)):
        true_id = int(y_true[i])
        pred_id = int(y_pred[i])

        rows.append({
            "case_text": test_df.iloc[i][cfg.text_col],
            "true_label": id2label[true_id],
            "predicted_label": id2label[pred_id],
            "top1_probability": float(probs[i, pred_id]),
            f"top{cfg.top_k}_labels": " | ".join(
                id2label[int(idx)]
                for idx in topk_indices[i]
            ),
            f"top{cfg.top_k}_probabilities": " | ".join(
                f"{float(prob):.6f}"
                for prob in topk_probs[i]
            ),
            "correct_top1": bool(true_id == pred_id),
            f"correct_top{cfg.top_k}": bool(true_id in topk_indices[i]),
        })

    pd.DataFrame(rows).to_csv(
        cfg.output_path / "test_predictions.csv",
        index=False,
    )

    report = classification_report(
        y_true,
        y_pred,
        target_names=[
            id2label[i]
            for i in range(num_labels)
        ],
        digits=4,
        zero_division=0,
    )

    with open(
        cfg.output_path / "classification_report.txt",
        "w",
        encoding="utf-8",
    ) as f:
        f.write(report)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(num_labels),
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            id2label[i]
            for i in range(num_labels)
        ],
        columns=[
            id2label[i]
            for i in range(num_labels)
        ],
    )

    cm_df.to_csv(cfg.output_path / "confusion_matrix.csv")

    plt.figure(figsize=(18, 16))
    plt.imshow(cm, interpolation="nearest")
    plt.title("BERT 512 Confusion Matrix")
    plt.colorbar()
    plt.xticks(
        np.arange(num_labels),
        [id2label[i] for i in range(num_labels)],
        rotation=90,
        fontsize=7,
    )
    plt.yticks(
        np.arange(num_labels),
        [id2label[i] for i in range(num_labels)],
        fontsize=7,
    )
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.savefig(cfg.output_path / "confusion_matrix.png", dpi=300)
    plt.close()

    return metrics


def main():
    cfg = CFG

    set_reproducibility(cfg.random_state)

    train_df, valid_df, test_df = load_splits(cfg)

    label2id, id2label = build_label_mappings(
        train_df,
        valid_df,
        test_df,
        cfg,
    )

    with open(
        cfg.output_path / "label_mapping.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "label2id": label2id,
                "id2label": {
                    str(k): v
                    for k, v in id2label.items()
                },
            },
            f,
            indent=4,
            ensure_ascii=False,
        )

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

    all_df = pd.concat(
        [train_df, valid_df, test_df],
        axis=0,
        ignore_index=True,
    )

    save_token_length_stats(all_df, tokenizer, cfg)

    train_labels = train_df[cfg.label_col].map(label2id).tolist()
    valid_labels = valid_df[cfg.label_col].map(label2id).tolist()
    test_labels = test_df[cfg.label_col].map(label2id).tolist()

    train_dataset = ClinicalTextDataset(
        train_df[cfg.text_col].tolist(),
        train_labels,
        tokenizer,
        cfg.max_length,
    )

    valid_dataset = ClinicalTextDataset(
        valid_df[cfg.text_col].tolist(),
        valid_labels,
        tokenizer,
        cfg.max_length,
    )

    test_dataset = ClinicalTextDataset(
        test_df[cfg.text_col].tolist(),
        test_labels,
        tokenizer,
        cfg.max_length,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=len(label2id),
        id2label={
            int(k): v
            for k, v in id2label.items()
        },
        label2id=label2id,
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    training_args = build_training_args(cfg)

    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_dataset,
        "eval_dataset": valid_dataset,
        "data_collator": data_collator,
        "compute_metrics": compute_metrics_builder(
            cfg,
            num_labels=len(label2id),
        ),
        "callbacks": [
            EarlyStoppingCallback(
                early_stopping_patience=cfg.early_stopping_patience,
            )
        ],
    }

    trainer_params = inspect.signature(Trainer.__init__).parameters

    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    logger.info("Starting BERT 512 fine-tuning...")
    trainer.train()

    logger.info("Evaluating best model on validation split...")
    valid_metrics = trainer.evaluate(valid_dataset)

    with open(
        cfg.output_path / "valid_metrics.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(valid_metrics, f, indent=4, ensure_ascii=False)

    logger.info("Running final evaluation on test split...")
    test_metrics = save_test_outputs(
        trainer,
        test_dataset,
        test_df,
        id2label,
        cfg,
    )

    save_training_history(trainer, cfg)

    trainer.save_model(cfg.output_path / "best_model")
    tokenizer.save_pretrained(cfg.output_path / "best_model")

    summary = {
        "experiment": "BERT 512 baseline",
        "model_name": cfg.model_name,
        "max_length": cfg.max_length,
        "num_train_epochs": cfg.num_train_epochs,
        "learning_rate": cfg.learning_rate,
        "weight_decay": cfg.weight_decay,
        "early_stopping_patience": cfg.early_stopping_patience,
        "random_state": cfg.random_state,
        "num_classes": len(label2id),
        "train_rows": len(train_df),
        "valid_rows": len(valid_df),
        "test_rows": len(test_df),
        "valid_metrics": valid_metrics,
        "test_metrics": test_metrics,
    }

    with open(
        cfg.output_path / "experiment_summary.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(summary, f, indent=4, ensure_ascii=False)

    logger.info("DONE.")
    logger.info("Outputs saved in: %s", cfg.output_path)


if __name__ == "__main__":
    main()

Writing experiments/06_bert_512_baseline.py


In [ ]:
!python experiments/06_bert_512_baseline.py

13:12:53 [INFO] Train rows: 1927
13:12:53 [INFO] Valid rows: 413
13:12:53 [INFO] Test rows: 414
13:12:53 [INFO] Number of classes: 43
13:12:54 [INFO] HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
13:12:54 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
13:12:54 [INFO] HTTP Request: GET https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
config.json: 100% 570/570 [00:00<00:00, 2.89MB/s]
13:12:54 [INFO] HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
13:12:54 [INFO] HTTP Request: GET https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 293kB/s]
13:12:54 [INFO] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/addition

In [ ]:
import json

with open("outputs/06_bert_512_baseline/test_metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)

metrics

{'test_accuracy': 0.5917874396135265,
 'test_macro_f1': 0.4637563930369323,
 'test_weighted_f1': 0.5490110237424927,
 'test_top5_accuracy': 0.8381642512077294}